In [ ]:
import signal, time as _t
_T0 = _t.time()
def _cap(*a): raise TimeoutError("25 min cap hit after %.0fs" % (_t.time() - _T0))
signal.signal(signal.SIGALRM, _cap); signal.alarm(25 * 60)
import subprocess, os, json, time
def sh(c): print(c, flush=True); subprocess.run(c, shell=True, check=True)
sh("pip -q install 'transformers==4.57.1' diffusers==0.35.2 gguf>=0.10 sentencepiece protobuf")
import torch, numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download, snapshot_download
from diffusers import FluxPipeline, FluxTransformer2DModel, GGUFQuantizationConfig, FlowMatchEulerDiscreteScheduler
gguf = hf_hub_download("city96/FLUX.1-dev-gguf", "flux1-dev-Q4_K_S.gguf")
BASE = snapshot_download("ostris/Flex.1-alpha", allow_patterns=["model_index.json","text_encoder/*","text_encoder_2/*","tokenizer/*","tokenizer_2/*","vae/*","scheduler/*"])
CFG = "/kaggle/working/flux_dev_cfg"; os.makedirs(f"{CFG}/transformer", exist_ok=True)
json.dump({"_class_name":"FluxTransformer2DModel","attention_head_dim":128,"axes_dims_rope":[16,56,56],"guidance_embeds":True,
           "in_channels":64,"joint_attention_dim":4096,"num_attention_heads":24,"num_layers":19,"num_single_layers":38,
           "patch_size":1,"pooled_projection_dim":768}, open(f"{CFG}/transformer/config.json","w"))
sched = FlowMatchEulerDiscreteScheduler(num_train_timesteps=1000, shift=3.0, use_dynamic_shifting=True, base_shift=0.5,
                                        max_shift=1.15, base_image_seq_len=256, max_image_seq_len=4096)
D0, D1, DT = "cuda:0", "cuda:1", torch.float16
PROMPT = "an impressionist oil painting of a campfire in a pine forest at night, full moon"
tp = FluxPipeline.from_pretrained(BASE, transformer=None, vae=None, torch_dtype=DT).to(D1)
with torch.inference_mode():
    pe, ppe, _ = tp.encode_prompt(PROMPT, prompt_2=None, device=D1, max_sequence_length=256)
print("prompt embeds finite:", bool(torch.isfinite(pe).all()), bool(torch.isfinite(ppe).all()), flush=True)
pe, ppe = pe.to(D0), ppe.to(D0); del tp; torch.cuda.empty_cache()
os.makedirs("/kaggle/working/flux_out", exist_ok=True)

def load():
    t = FluxTransformer2DModel.from_single_file(gguf, config=CFG, subfolder="transformer",
            quantization_config=GGUFQuantizationConfig(compute_dtype=DT), torch_dtype=DT)
    return t
nonfinite = {"n": 0}
def watch(m, i, o):
    x = o[0] if isinstance(o, tuple) else o
    if torch.is_tensor(x) and not torch.isfinite(x).all(): nonfinite["n"] += 1

def run(name, t):
    pipe = FluxPipeline.from_pretrained(BASE, transformer=t, text_encoder=None, text_encoder_2=None, tokenizer=None,
                                        tokenizer_2=None, scheduler=sched, torch_dtype=DT).to(D0)
    pipe.vae.to(torch.float32)
    hs = [b.register_forward_hook(watch) for b in list(t.transformer_blocks) + list(t.single_transformer_blocks)]
    nonfinite["n"] = 0; torch.cuda.reset_peak_memory_stats(0); t0 = time.time()
    try:
        with torch.inference_mode():
            lat = pipe(prompt_embeds=pe, pooled_prompt_embeds=ppe, height=512, width=512, num_inference_steps=16,
                       guidance_scale=3.5, output_type="latent", generator=torch.Generator(D0).manual_seed(7)).images
            lat = pipe._unpack_latents(lat, 512, 512, pipe.vae_scale_factor).float()
            lat = lat / pipe.vae.config.scaling_factor + pipe.vae.config.shift_factor
            img = pipe.vae.decode(lat).sample
        arr = ((img[0].permute(1, 2, 0).clamp(-1, 1) + 1) * 127.5).byte().cpu().numpy()
        Image.fromarray(arr).save(f"/kaggle/working/flux_out/{name}.png")
        print(f"{name}: {time.time()-t0:.0f}s, non-finite block outputs {nonfinite['n']}, pixel std {arr.std():.1f}, "
              f"peak GiB {torch.cuda.max_memory_allocated(0)/2**30:.1f}", flush=True)
    except BaseException as e:
        import traceback; traceback.print_exc(); print("FAILED", name, flush=True)
        if isinstance(e, TimeoutError): raise
    finally:
        for h in hs: h.remove()
    del pipe; torch.cuda.empty_cache()

t = load().to(D0)
run("A_plain_fp16", t)
exec(compile('"""Run the Qwen-Image(-Layered) transformer on GPUs without bf16 (Kaggle T4, P100, RTX 20xx) without NaNs.\n\nQwen-Image is trained for bf16; its hidden values exceed the fp16 limit (65504), so a plain fp16 run gives all-NaN\nlatents (ComfyUI also lists Qwen-Image as bf16/fp32 only). This keeps the fast fp16 tensor-core matmuls but holds\neverything else in fp32:\n\n  * every Linear: input scaled down to absmax 32, fp16 matmul, result scaled back in fp32\n    (falls back to a fp32 matmul for that call if the fp16 result is still non-finite)\n  * attention: q/k in fp16 (RMS-normalised, small), v scaled the same way (attention is linear in v)\n  * residual stream, norms, gates, embeddings: fp32\n  * the model\'s output is returned in fp16, so the diffusers pipeline is unchanged\n\nMeasured on Kaggle 2x T4 (Q4_K_S GGUF, 640 px, 4 layers): no NaN, 42 s/step, peak 8.8 / 8.4 GB.\n\n    from layerproto.fp16_mixed import apply_fp16_mixed\n    apply_fp16_mixed(transformer)        # once, after loading; idempotent\n"""\nimport types\n\nimport torch\nimport torch.nn.functional as F\n\nLINEAR_LIMIT = 32.0  # absmax the fp16 matmul input is scaled to (1024 was still too high: NaNs)\nATTN_LIMIT = 32.0\n\n\ndef _dequant(w):\n    if type(w).__name__ == "GGUFParameter":\n        from diffusers.quantizers.gguf.utils import dequantize_gguf_tensor\n        return dequantize_gguf_tensor(w)\n    return w\n\n\ndef _mixed_linear(self, x):\n    x = x.float()\n    s = (x.abs().amax() / LINEAR_LIMIT).clamp_(min=1.0)\n    y = self._fp16_orig_forward((x / s).to(torch.float16)).float()   # = W x/s + b\n    b = getattr(self, "bias", None)\n    if not torch.isfinite(y).all():                                  # rare: this call in fp32\n        y = F.linear(x, _dequant(self.weight).float(), None if b is None else b.float())\n    elif b is None:\n        y = y * s\n    else:\n        b = b.float()\n        y = (y - b) * s + b\n    lora = getattr(self, "_fp16_lora", None)\n    if lora is not None:                                             # low-rank delta, same input scaling\n        A, B, k = lora\n        if A.device != x.device:                                     # module was moved (e.g. split over GPUs)\n            A, B = A.to(x.device), B.to(x.device)\n            self._fp16_lora = (A, B, k)\n        xs = (x / s).to(torch.float16)\n        y = y + F.linear(F.linear(xs, A), B).float() * (s * k)\n    return y\n\n\ndef _lora_pairs(sd):\n    """{module name: (A [r, in], B [out, r], alpha or None)} for common LoRA key layouts (PEFT / kohya / diffusers)."""\n    out = {}\n    for key in sd:\n        for a_suf, b_suf in ((".lora_A.weight", ".lora_B.weight"), (".lora_down.weight", ".lora_up.weight"),\n                             (".lora_A.default.weight", ".lora_B.default.weight")):\n            if key.endswith(a_suf):\n                base = key[: -len(a_suf)]\n                bkey = base + b_suf\n                if bkey in sd:\n                    alpha = sd.get(base + ".alpha")\n                    name = base\n                    for pre in ("base_model.model.", "diffusion_model.", "transformer.", "model."):\n                        if name.startswith(pre):\n                            name = name[len(pre):]\n                    out[name] = (sd[key], sd[bkey], None if alpha is None else float(alpha))\n    return out\n\n\ndef add_lora(model, path_or_state_dict, scale: float = 1.0):\n    """Attach a LoRA to an fp16_mixed-patched model (works with GGUF-quantised linears, nothing is merged).\n    Returns (matched modules, LoRA modules not found)."""\n    sd = path_or_state_dict\n    if isinstance(sd, str):\n        from safetensors.torch import load_file\n        sd = load_file(sd)\n    pairs = _lora_pairs(sd)\n    mods = dict(model.named_modules())\n    hit, miss = 0, []\n    for name, (A, B, alpha) in pairs.items():\n        m = mods.get(name)\n        if m is None or not hasattr(m, "_fp16_orig_forward"):\n            miss.append(name)\n            continue\n        dev = next((p.device for p in m.parameters()), torch.device("cpu"))\n        r = A.shape[0]\n        k = scale * ((alpha / r) if alpha else 1.0)\n        m._fp16_lora = (A.to(dev, torch.float16), B.to(dev, torch.float16), k)\n        hit += 1\n    return hit, miss\n\n\ndef remove_lora(model):\n    for _, m in model.named_modules():\n        if hasattr(m, "_fp16_lora"):\n            del m._fp16_lora\n\n\ndef _patch_attention():\n    import diffusers.models.transformers.transformer_qwenimage as tq\n    if getattr(tq, "_fp16_orig_attn", None) is not None:\n        return\n    orig = tq.dispatch_attention_fn\n\n    def attn_fp16(q, k, v, attn_mask=None, **kw):\n        s = (v.float().abs().amax() / ATTN_LIMIT).clamp_(min=1.0)\n        if attn_mask is not None and attn_mask.is_floating_point():\n            attn_mask = attn_mask.to(torch.float16)\n        o = orig(q.to(torch.float16), k.to(torch.float16), (v.float() / s).to(torch.float16), attn_mask=attn_mask, **kw)\n        return o.float() * s\n\n    tq._fp16_orig_attn = orig\n    tq.dispatch_attention_fn = attn_fp16\n\n\ndef apply_fp16_mixed(model):\n    """Patch a loaded QwenImageTransformer2DModel in place. Returns (linears patched, params moved to fp32)."""\n    _patch_attention()\n    n_lin = n_par = 0\n    for _, m in model.named_modules():\n        if "Linear" in type(m).__name__:\n            if not hasattr(m, "_fp16_orig_forward"):\n                m._fp16_orig_forward = m.forward\n            m.forward = types.MethodType(_mixed_linear, m)\n            n_lin += 1\n        else:\n            for _, p in m.named_parameters(recurse=False):\n                if p.dtype == torch.float16:\n                    p.data = p.data.float()\n                    n_par += 1\n    if not hasattr(model, "_fp16_orig_model_forward"):\n        model._fp16_orig_model_forward = model.forward\n\n        def fwd(*a, **k):\n            o = model._fp16_orig_model_forward(*a, **k)\n            if isinstance(o, tuple):\n                return (o[0].to(torch.float16),) + tuple(o[1:])\n            if hasattr(o, "sample"):\n                o.sample = o.sample.to(torch.float16)\n            return o\n\n        model.forward = fwd\n    return n_lin, n_par\n', "fp16_mixed", "exec"))
apply_fp16_mixed(t)
run("B_fp16_mixed", t)
print("total", round(time.time() - _T0), "s", flush=True)